# Extract Subject-Level Acoustic Tracking Features

Use this notebook after Eelbrain-main TRF outputs have been generated. It reads cached TRF outputs and summarizes tracking into subject-level feature columns.


In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

def find_pipeline_dir(start=Path.cwd()):
    start = Path(start).resolve()
    candidates = [start, *start.parents, start / 'analysis' / 'trf_pipeline']
    for path in candidates:
        if (path / 'alice_eelbrain_main_experiment.py').exists():
            return path
    raise FileNotFoundError(f'Could not find alice_eelbrain_main_experiment.py from {start}')


PIPELINE_DIR = find_pipeline_dir()
if str(PIPELINE_DIR) not in sys.path:
    sys.path.insert(0, str(PIPELINE_DIR))

from alice_eelbrain_main_experiment import BIDS_ROOT, TRF_OPTIONS, alice

MODEL = 'gammatone-8'
STATE = {'raw': '0.5-20', 'epoch': 'story-segments', 'inv': ''}
FEATURE_DIR = Path('/Users/yanyuwoo/Data/Alice Comprehension/features')
QC_DIR = Path('/Users/yanyuwoo/Data/Alice Comprehension/qc')
FEATURE_DIR.mkdir(parents=True, exist_ok=True)
QC_DIR.mkdir(parents=True, exist_ok=True)

subjects = alice.get_field_values('subject')
print(f'Subjects: {len(subjects)}')
print(f'Model: {MODEL}')


## Check Which TRF Outputs Exist

This does not fit any model. It only checks cache paths.

In [ ]:
cache_rows = []
for subject in subjects:
    path = Path(alice.load_trf(MODEL, subject=subject, path_only=True, **STATE, **TRF_OPTIONS))
    cache_rows.append({'subject': f'sub-{subject}', 'trf_path': str(path), 'exists': path.exists()})

cache_report = pd.DataFrame(cache_rows)
display(cache_report['exists'].value_counts(dropna=False).rename('n_subjects'))
display(cache_report.head())
cache_report.to_csv(QC_DIR / 'trf_gammatone8_cache_status.csv', index=False)


## Inspect One Available Output

Run this before aggregating. It shows which metrics Eelbrain main produced for the first available subject.


In [ ]:
available = cache_report.loc[cache_report['exists'], 'subject'].str.replace('sub-', '', regex=False).tolist()
if not available:
    print('No cached TRF outputs found yet. Run notebook 02 for one subject or notebook 03 for batch first.')
else:
    example_subject = available[0]
    ds = alice.load_trfs(example_subject, MODEL, **STATE, **TRF_OPTIONS)
    print(f'Example subject: sub-{example_subject}')
    print(ds)
    print('
Dataset keys:')
    print(list(ds.keys()))


## Aggregate Tracking Scores

`tracking_r_mean` is the main subject-level feature. It averages the TRF validation correlation across available `r` values. `tracking_r_best` is diagnostic only because max scores are more sensitive to leakage and noisy channels.

In [ ]:
def numeric_values(value):
    arr = np.asarray(value, dtype=float)
    return arr[np.isfinite(arr)]


feature_rows = []
failed_rows = []

for subject in subjects:
    subject_label = f'sub-{subject}'
    try:
        path = Path(alice.load_trf(MODEL, subject=subject, path_only=True, **STATE, **TRF_OPTIONS))
        if not path.exists():
            failed_rows.append({'subject': subject_label, 'reason': 'missing_trf_cache', 'path': str(path)})
            continue

        ds = alice.load_trfs(subject, MODEL, **STATE, **TRF_OPTIONS)
        if 'r' not in ds:
            failed_rows.append({'subject': subject_label, 'reason': 'dataset_has_no_r', 'path': str(path)})
            continue

        r_values = numeric_values(ds['r'])
        if len(r_values) == 0:
            failed_rows.append({'subject': subject_label, 'reason': 'empty_r_values', 'path': str(path)})
            continue

        feature_rows.append({
            'subject': subject_label,
            'predictor_name': MODEL,
            'tracking_r_mean': float(np.mean(r_values)),
            'tracking_r_median': float(np.median(r_values)),
            'tracking_r_best': float(np.max(r_values)),
            'tracking_r_min': float(np.min(r_values)),
            'n_r_values': int(len(r_values)),
            'trf_path': str(path),
            'status': 'ok',
        })
    except Exception as exc:
        failed_rows.append({'subject': subject_label, 'reason': repr(exc), 'path': ''})

features = pd.DataFrame(feature_rows)
qc = pd.DataFrame(failed_rows)

feature_path = FEATURE_DIR / 'trf_gammatone8_acoustic_tracking.csv'
qc_path = QC_DIR / 'trf_gammatone8_feature_extraction_qc.csv'
features.to_csv(feature_path, index=False)
qc.to_csv(qc_path, index=False)

print(f'Wrote features: {feature_path}')
print(f'Wrote QC: {qc_path}')
print(f'Feature rows: {len(features)}')
print(f'QC rows: {len(qc)}')
display(features.head())
display(qc.head())
